# Case Técnico — EDA BigQuery-first e preparação para previsão diária

Este notebook simula uma arquitetura mais próxima de produção:

**BigQuery / SQL → Data Quality, filtros, agregações e regras em escala**  
**Python → estatística, visualização, interpretação e Machine Learning**

Essa separação evita trazer granularidade desnecessária para memória, aproveita particionamento/clustering e deixa explícito o domínio de SQL e BigQuery.

Objetivos do case:
- avaliar qualidade e consistência dos dados;
- estudar preço médio de venda;
- medir potencial de vendas por dia da semana;
- entender flutuações ao longo do dia;
- preparar a série diária para previsão.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gb_ml.io import estimate_query_bytes, read_query, table_metadata

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ID = "gms-prod-01"
BQ_LOCATION = "us-east4"
TABLE_ID = "gms-prod-01.projeto_gb.fact_vendas"
STG_TABLE_ID = "gms-prod-01.projeto_gb.stg_fact_vendas"

START_DATE = "2025-11-01"
END_DATE = "2026-06-30"

DATE_FILTER = f"DATE(dt_hr_venda) BETWEEN DATE '{START_DATE}' AND DATE '{END_DATE}'"

## Helper de execução

Antes de cada consulta, fazemos um **dry-run** para estimar bytes processados. Mesmo em um case pequeno, isso demonstra uma prática importante de custo/performance no BigQuery.

In [ ]:
def run_bq(sql: str, label: str | None = None) -> pd.DataFrame:
    bytes_processed = estimate_query_bytes(
        sql,
        project_id=PROJECT_ID,
        location=BQ_LOCATION,
    )
    mb = bytes_processed / 1024**2
    print(f"[BigQuery] {label or 'query'} | estimativa processada: {mb:,.2f} MB")

    return read_query(
        sql,
        project_id=PROJECT_ID,
        location=BQ_LOCATION,
    )

# 1. Data Understanding — fonte, schema e período

In [ ]:
metadata, bq_schema = table_metadata(
    TABLE_ID,
    project_id=PROJECT_ID,
    location=BQ_LOCATION,
)

display(pd.DataFrame([metadata]))
display(bq_schema)

In [ ]:
sql_overview = f"""
SELECT
  COUNT(*) AS linhas,
  MIN(dt_hr_venda) AS dt_min,
  MAX(dt_hr_venda) AS dt_max,
  COUNT(DISTINCT DATE(dt_hr_venda)) AS dias,
  COUNT(DISTINCT des_canal_venda_final_agrup) AS canais,
  COUNT(DISTINCT des_categoria_material) AS categorias_brutas
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

overview = run_bq(sql_overview, "overview")
display(overview)

### 1.1 Precisão monetária no BigQuery

Na carga original foram encontrados números monetários com artefatos de ponto flutuante, por exemplo `3731.5600000000004`.

**Decisão de engenharia:** preservar a carga bruta em `stg_fact_vendas` e usar `NUMERIC` arredondado a 2 casas em `fact_vendas`. Não usamos `BIGNUMERIC`, pois as casas adicionais representam ruído de representação e não precisão financeira útil.

In [ ]:
sql_precision = f"""
SELECT
  COUNT(*) AS linhas,
  COUNTIF(receita_aprovada != ROUND(receita_aprovada, 2)) AS receita_acima_2_casas,
  COUNTIF(vlr_venda_desconto != ROUND(vlr_venda_desconto, 2)) AS desconto_acima_2_casas
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

precision_check = run_bq(sql_precision, "precisão monetária")
display(precision_check)

display(
    bq_schema[
        bq_schema["name"].isin(["receita_aprovada", "vlr_venda_desconto"])
    ]
)

### 1.2 Impacto financeiro da normalização para 2 casas

A `stg_fact_vendas` preserva a carga original. Aqui quantificamos o efeito da normalização monetária.

**Importante:** a fact tratada usa **arredondamento (`ROUND`)**, não truncamento. Mostramos os dois cenários para evidenciar o impacto e o possível viés introduzido por truncar.

In [ ]:
sql_precision_impact = f"""
WITH base AS (
  SELECT
    SAFE_CAST(receita_aprovada AS BIGNUMERIC) AS receita_original
  FROM `{STG_TABLE_ID}`
  WHERE {DATE_FILTER}
    AND receita_aprovada IS NOT NULL
),
calc AS (
  SELECT
    receita_original,
    ROUND(receita_original, 2) AS receita_round_2,
    TRUNC(receita_original, 2) AS receita_trunc_2
  FROM base
)
SELECT
  CAST(SUM(receita_original) AS FLOAT64) AS receita_original_total,
  CAST(SUM(receita_round_2) AS FLOAT64) AS receita_round_2_total,
  CAST(SUM(receita_trunc_2) AS FLOAT64) AS receita_trunc_2_total,

  CAST(SUM(receita_original - receita_round_2) AS FLOAT64)
    AS diferenca_liquida_round,

  CAST(SUM(ABS(receita_original - receita_round_2)) AS FLOAT64)
    AS ajuste_absoluto_total_round,

  CAST(MAX(ABS(receita_original - receita_round_2)) AS FLOAT64)
    AS maior_ajuste_linha_round,

  SAFE_DIVIDE(
    CAST(ABS(SUM(receita_original - receita_round_2)) AS FLOAT64),
    CAST(ABS(SUM(receita_original)) AS FLOAT64)
  ) AS pct_impacto_liquido_round,

  CAST(SUM(receita_original - receita_trunc_2) AS FLOAT64)
    AS diferenca_liquida_trunc,

  CAST(SUM(ABS(receita_original - receita_trunc_2)) AS FLOAT64)
    AS ajuste_absoluto_total_trunc,

  SAFE_DIVIDE(
    CAST(ABS(SUM(receita_original - receita_trunc_2)) AS FLOAT64),
    CAST(ABS(SUM(receita_original)) AS FLOAT64)
  ) AS pct_impacto_liquido_trunc

FROM calc
"""

precision_impact = run_bq(
    sql_precision_impact,
    "impacto da normalização monetária",
)
display(precision_impact.T.rename(columns={0: "valor"}))

print(
    "Impacto líquido do ROUND sobre a receita total:",
    f"{float(precision_impact.loc[0, 'pct_impacto_liquido_round']):.10%}",
)

# 2. Data Quality — checks executados no BigQuery

## 2.1 Nulos, domínios e valores impossíveis

Aqui o BigQuery faz o papel de camada de observabilidade: o Python recebe apenas o resumo.

In [ ]:
sql_quality = f"""
SELECT
  COUNT(*) AS linhas,

  COUNTIF(dt_hr_venda IS NULL) AS dt_hr_venda_nulos,
  COUNTIF(des_canal_venda_final_agrup IS NULL) AS canal_nulos,
  COUNTIF(des_categoria_material IS NULL) AS categoria_nulos,
  COUNTIF(receita_aprovada IS NULL) AS receita_nulos,
  COUNTIF(nr_pedidos IS NULL) AS pedidos_nulos,
  COUNTIF(qt_material IS NULL) AS itens_nulos,
  COUNTIF(vlr_venda_desconto IS NULL) AS desconto_nulos,

  COUNTIF(
    des_canal_venda_final_agrup NOT IN ('App', 'Site')
    OR des_canal_venda_final_agrup IS NULL
  ) AS canal_fora_dominio,

  COUNTIF(nr_pedidos <= 0) AS pedidos_nao_positivos,
  COUNTIF(qt_material <= 0) AS itens_nao_positivos,

  COUNTIF(receita_aprovada < 0) AS receita_negativa,
  COUNTIF(receita_aprovada = 0) AS receita_zero

FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

quality = run_bq(sql_quality, "data quality geral")
display(quality.T.rename(columns={0: "valor"}))

## 2.2 Duplicidade no grão

O grão descrito no case é **hora × canal × categoria**. Duplicidade nesse nível poderia inflar receita e demanda silenciosamente.

In [ ]:
sql_duplicates = f"""
WITH duplicados AS (
  SELECT
    dt_hr_venda,
    des_canal_venda_final_agrup,
    des_categoria_material,
    COUNT(*) AS qtd
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1, 2, 3
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*) AS grupos_duplicados,
  COALESCE(SUM(qtd - 1), 0) AS linhas_excedentes
FROM duplicados
"""

duplicates = run_bq(sql_duplicates, "duplicidade no grão")
display(duplicates)

## 2.3 Categoria inválida

A coluna deveria conter categorias de produto, mas há registros numéricos entre 0 e 1. Em SQL, `SAFE_CAST` permite identificá-los sem quebrar a consulta.

**Decisão:** esses registros permanecem nos totais de venda/demanda, mas são excluídos apenas de análises que dependem da categoria correta.

In [ ]:
sql_invalid_category = f"""
SELECT
  COUNT(*) AS linhas,
  COUNTIF(SAFE_CAST(des_categoria_material AS NUMERIC) IS NOT NULL) AS categorias_invalidas,
  SAFE_DIVIDE(
    COUNTIF(SAFE_CAST(des_categoria_material AS NUMERIC) IS NOT NULL),
    COUNT(*)
  ) AS pct_categoria_invalida,
  COUNT(DISTINCT IF(
    SAFE_CAST(des_categoria_material AS NUMERIC) IS NULL,
    des_categoria_material,
    NULL
  )) AS categorias_validas_distintas
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

invalid_category = run_bq(sql_invalid_category, "categorias inválidas")
display(invalid_category)

### Exemplos das categorias inválidas

A consulta abaixo materializa os valores mais frequentes classificados como numéricos. O objetivo é tornar visível que o problema não é uma categoria de negócio rara, mas valores incompatíveis com a semântica da coluna.

In [ ]:
sql_invalid_category_top10 = f"""
SELECT
  des_categoria_material AS categoria_invalida,
  COUNT(*) AS linhas,
  SUM(qt_material) AS itens,
  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
  AND SAFE_CAST(des_categoria_material AS NUMERIC) IS NOT NULL
GROUP BY 1
ORDER BY linhas DESC, categoria_invalida
LIMIT 10
"""

invalid_category_top10 = run_bq(
    sql_invalid_category_top10,
    "top 10 categorias inválidas",
)
display(invalid_category_top10)

In [ ]:
sql_invalid_category_month = f"""
SELECT
  FORMAT_DATE('%Y-%m', DATE(dt_hr_venda)) AS mes,
  COUNT(*) AS linhas,
  COUNTIF(SAFE_CAST(des_categoria_material AS NUMERIC) IS NOT NULL) AS categorias_invalidas,
  SAFE_DIVIDE(
    COUNTIF(SAFE_CAST(des_categoria_material AS NUMERIC) IS NOT NULL),
    COUNT(*)
  ) AS pct_invalida
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
GROUP BY 1
ORDER BY 1
"""

invalid_category_month = run_bq(
    sql_invalid_category_month,
    "categoria inválida por mês",
)
display(invalid_category_month)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    invalid_category_month["mes"],
    invalid_category_month["pct_invalida"] * 100,
    marker="o",
)
ax.set_title("Categoria inválida ao longo do tempo")
ax.set_xlabel("Mês")
ax.set_ylabel("% das linhas")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 2.4 Receita negativa

Receita negativa pode representar estorno, devolução ou ajuste. Sem confirmação de negócio, o comportamento correto é **quantificar e investigar**, não excluir automaticamente.

In [ ]:
sql_negative_revenue = f"""
SELECT
  COUNTIF(receita_aprovada < 0) AS linhas_receita_negativa,
  SAFE_DIVIDE(COUNTIF(receita_aprovada < 0), COUNT(*)) AS pct_linhas_negativas,
  CAST(SUM(IF(receita_aprovada < 0, receita_aprovada, 0)) AS FLOAT64) AS receita_negativa_total,
  CAST(SUM(IF(receita_aprovada >= 0, receita_aprovada, 0)) AS FLOAT64) AS receita_nao_negativa_total,
  SAFE_DIVIDE(
    -CAST(SUM(IF(receita_aprovada < 0, receita_aprovada, 0)) AS FLOAT64),
    CAST(SUM(IF(receita_aprovada >= 0, receita_aprovada, 0)) AS FLOAT64)
  ) AS impacto_abs_negativo_sobre_positivo
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

negative_revenue = run_bq(sql_negative_revenue, "receita negativa")
display(negative_revenue)

## 2.5 Cobertura temporal

O dataset deveria cobrir todas as horas entre 01/11/2025 e 30/06/2026. Geramos o calendário esperado no BigQuery e comparamos com o observado.

In [ ]:
sql_coverage = f"""
WITH horas_esperadas AS (
  SELECT DATETIME(ts) AS dt_hr_venda
  FROM UNNEST(
    GENERATE_TIMESTAMP_ARRAY(
      TIMESTAMP('{START_DATE} 00:00:00'),
      TIMESTAMP('{END_DATE} 23:00:00'),
      INTERVAL 1 HOUR
    )
  ) AS ts
),
horas_observadas AS (
  SELECT DISTINCT dt_hr_venda
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
),
canais AS (
  SELECT canal
  FROM UNNEST(['App', 'Site']) AS canal
),
hora_canal_observado AS (
  SELECT DISTINCT
    dt_hr_venda,
    des_canal_venda_final_agrup AS canal
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
),
faltas_hora_canal AS (
  SELECT h.dt_hr_venda, c.canal
  FROM horas_esperadas h
  CROSS JOIN canais c
  LEFT JOIN hora_canal_observado o
    ON h.dt_hr_venda = o.dt_hr_venda
   AND c.canal = o.canal
  WHERE o.dt_hr_venda IS NULL
)
SELECT
  (SELECT COUNT(*) FROM horas_esperadas) AS horas_esperadas,
  (
    SELECT COUNT(*)
    FROM horas_esperadas h
    LEFT JOIN horas_observadas o USING (dt_hr_venda)
    WHERE o.dt_hr_venda IS NULL
  ) AS horas_sem_registro,
  (SELECT COUNT(*) FROM faltas_hora_canal) AS combinacoes_hora_canal_ausentes
"""

coverage = run_bq(sql_coverage, "cobertura temporal")
display(coverage)

### Combinações hora × canal ausentes

Além da contagem, mostramos as combinações faltantes para entender se são eventos isolados ou um padrão operacional.

In [ ]:
sql_missing_hour_channel = f"""
WITH horas_esperadas AS (
  SELECT DATETIME(ts) AS dt_hr_venda
  FROM UNNEST(
    GENERATE_TIMESTAMP_ARRAY(
      TIMESTAMP('{START_DATE} 00:00:00'),
      TIMESTAMP('{END_DATE} 23:00:00'),
      INTERVAL 1 HOUR
    )
  ) AS ts
),
canais AS (
  SELECT canal
  FROM UNNEST(['App', 'Site']) AS canal
),
observado AS (
  SELECT DISTINCT
    dt_hr_venda,
    des_canal_venda_final_agrup AS canal
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
)
SELECT
  h.dt_hr_venda,
  c.canal
FROM horas_esperadas h
CROSS JOIN canais c
LEFT JOIN observado o
  ON h.dt_hr_venda = o.dt_hr_venda
 AND c.canal = o.canal
WHERE o.dt_hr_venda IS NULL
ORDER BY h.dt_hr_venda, c.canal
LIMIT 100
"""

missing_hour_channel = run_bq(
    sql_missing_hour_channel,
    "exemplos hora × canal ausentes",
)
display(missing_hour_channel)

print(f"Combinações exibidas: {len(missing_hour_channel)}")

# 3. Analytical EDA — BigQuery agrega, Python visualiza

## 3.1 Série diária para Analytics e Machine Learning

Somente aqui trazemos dados para Pandas: uma tabela diária de aproximadamente 242 linhas, em vez da granularidade horária completa.

### Por que não modelar diretamente no grão horário?

O arquivo horário é valioso e será explorado, mas o **target permanece diário** porque o case solicita previsão de venda diária.

O grão horário pode ser usado a nosso favor para:
- entender o perfil intradiário;
- criar, se necessário, features **históricas** de comportamento horário;
- criar lags por canal/categoria/hora.

Não devemos usar informações do próprio dia realizado para prever esse mesmo dia. Modelar hora a hora e depois somar seria possível, mas adicionaria complexidade sem responder melhor ao objetivo principal do case. A estratégia inicial, portanto, é **forecast diário + enriquecimento com sinais históricos da granularidade horária**.

In [ ]:
sql_daily = f"""
SELECT
  DATE(dt_hr_venda) AS data,

  CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
    WHEN 2 THEN 0
    WHEN 3 THEN 1
    WHEN 4 THEN 2
    WHEN 5 THEN 3
    WHEN 6 THEN 4
    WHEN 7 THEN 5
    WHEN 1 THEN 6
  END AS dia_semana_num,

  CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
    WHEN 2 THEN 'Segunda'
    WHEN 3 THEN 'Terça'
    WHEN 4 THEN 'Quarta'
    WHEN 5 THEN 'Quinta'
    WHEN 6 THEN 'Sexta'
    WHEN 7 THEN 'Sábado'
    WHEN 1 THEN 'Domingo'
  END AS dia_semana,

  FORMAT_DATE('%Y-%m', DATE(dt_hr_venda)) AS mes,
  EXTRACT(MONTH FROM DATE(dt_hr_venda)) AS mes_num,
  EXTRACT(YEAR FROM DATE(dt_hr_venda)) AS ano,
  EXTRACT(ISOWEEK FROM DATE(dt_hr_venda)) AS semana_ano,
  EXTRACT(ISOYEAR FROM DATE(dt_hr_venda)) AS ano_iso,
  DATE_TRUNC(DATE(dt_hr_venda), WEEK(MONDAY)) AS semana_inicio,
  IF(EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda)) IN (1, 7), 1, 0) AS fim_de_semana,

  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita_aprovada,
  SUM(nr_pedidos) AS nr_pedidos,
  SUM(qt_material) AS qt_material,
  CAST(SUM(vlr_venda_desconto) AS FLOAT64) AS vlr_venda_desconto,

  SAFE_DIVIDE(
    CAST(SUM(receita_aprovada) AS FLOAT64),
    SUM(qt_material)
  ) AS preco_medio_item,

  SAFE_DIVIDE(
    CAST(SUM(receita_aprovada) AS FLOAT64),
    SUM(nr_pedidos)
  ) AS ticket_medio,

  SAFE_DIVIDE(
    SUM(qt_material),
    SUM(nr_pedidos)
  ) AS itens_por_pedido,

  SAFE_DIVIDE(
    CAST(SUM(vlr_venda_desconto) AS FLOAT64),
    CAST(SUM(receita_aprovada + vlr_venda_desconto) AS FLOAT64)
  ) AS taxa_desconto

FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9
ORDER BY 1
"""

daily = run_bq(sql_daily, "agregação diária")
daily["data"] = pd.to_datetime(daily["data"])

print(f"Linhas trazidas para Pandas: {len(daily)}")
display(daily.head())
display(
    daily[
        [
            "receita_aprovada",
            "nr_pedidos",
            "qt_material",
            "preco_medio_item",
            "ticket_medio",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T
)

### Evolução da demanda

`qt_material` é a melhor proxy disponível para demanda operacional em unidades.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(daily["data"], daily["qt_material"], label="Itens vendidos", alpha=0.65)
ax.plot(
    daily["data"],
    daily["qt_material"].rolling(7, min_periods=1).mean(),
    label="Média móvel 7 dias",
)
ax.plot(
    daily["data"],
    daily["qt_material"].rolling(28, min_periods=1).mean(),
    label="Média móvel 28 dias",
)
ax.set_title("Demanda diária")
ax.set_xlabel("Data")
ax.set_ylabel("Itens vendidos")
ax.legend()
fig.tight_layout()
plt.show()

### Baseline imediato: médias móveis de 7 e 28 dias

As médias móveis do gráfico acima são descritivas. Para tratá-las como **previsões**, precisamos deslocá-las um dia para trás, garantindo que a previsão de *t* use apenas informação disponível até *t-1*.

Usamos uma janela comum após 28 dias para comparar os dois baselines de forma justa. A métrica principal é **WAPE**. Também mostramos `1 - WAPE` como uma leitura percentual intuitiva, mas ela não substitui as métricas formais do modelo.

In [ ]:
baseline = daily[["data", "qt_material"]].copy()

baseline["pred_media_movel_7"] = (
    baseline["qt_material"]
    .shift(1)
    .rolling(7, min_periods=7)
    .mean()
)

baseline["pred_media_movel_28"] = (
    baseline["qt_material"]
    .shift(1)
    .rolling(28, min_periods=28)
    .mean()
)

evaluation = baseline.dropna(
    subset=["pred_media_movel_7", "pred_media_movel_28"]
).copy()

def forecast_metrics(y_true, y_pred):
    error = y_pred - y_true
    abs_error = np.abs(error)

    mae = abs_error.mean()
    wape = abs_error.sum() / np.abs(y_true).sum()
    bias = error.sum() / y_true.sum()

    return {
        "MAE": mae,
        "WAPE": wape,
        "1-WAPE": 1 - wape,
        "Bias": bias,
    }

baseline_metrics = pd.DataFrame([
    {
        "modelo": "Média móvel 7 dias",
        **forecast_metrics(
            evaluation["qt_material"],
            evaluation["pred_media_movel_7"],
        ),
    },
    {
        "modelo": "Média móvel 28 dias",
        **forecast_metrics(
            evaluation["qt_material"],
            evaluation["pred_media_movel_28"],
        ),
    },
])

display(baseline_metrics)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(evaluation["data"], evaluation["qt_material"], label="Real", alpha=0.65)
ax.plot(evaluation["data"], evaluation["pred_media_movel_7"], label="MM 7 dias")
ax.plot(evaluation["data"], evaluation["pred_media_movel_28"], label="MM 28 dias")
ax.set_title("Backtest simples — médias móveis como baseline")
ax.set_xlabel("Data")
ax.set_ylabel("Itens vendidos")
ax.legend()
fig.tight_layout()
plt.show()

## 3.2 Outliers diários — thresholds calculados no BigQuery

A regra de Tukey/IQR é usada para **sinalizar dias extremos**, não para excluí-los. Em e-commerce, campanhas podem gerar picos legítimos.

In [ ]:
sql_daily_outliers = f"""
WITH daily AS (
  SELECT
    DATE(dt_hr_venda) AS data,
    SUM(qt_material) AS qt_material,
    CAST(SUM(receita_aprovada) AS FLOAT64) AS receita_aprovada
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1
),
quartis AS (
  SELECT
    APPROX_QUANTILES(qt_material, 100)[OFFSET(25)] AS q1_itens,
    APPROX_QUANTILES(qt_material, 100)[OFFSET(75)] AS q3_itens,
    APPROX_QUANTILES(receita_aprovada, 100)[OFFSET(25)] AS q1_receita,
    APPROX_QUANTILES(receita_aprovada, 100)[OFFSET(75)] AS q3_receita
  FROM daily
),
limites AS (
  SELECT
    *,
    q1_itens - 1.5 * (q3_itens - q1_itens) AS li_itens,
    q3_itens + 1.5 * (q3_itens - q1_itens) AS ls_itens,
    q1_receita - 1.5 * (q3_receita - q1_receita) AS li_receita,
    q3_receita + 1.5 * (q3_receita - q1_receita) AS ls_receita
  FROM quartis
)
SELECT
  l.*,
  COUNTIF(d.qt_material < l.li_itens OR d.qt_material > l.ls_itens) AS dias_outlier_itens,
  COUNTIF(d.receita_aprovada < l.li_receita OR d.receita_aprovada > l.ls_receita) AS dias_outlier_receita
FROM daily d
CROSS JOIN limites l
GROUP BY
  q1_itens, q3_itens, q1_receita, q3_receita,
  li_itens, ls_itens, li_receita, ls_receita
"""

outlier_limits = run_bq(sql_daily_outliers, "limites de outliers")
display(outlier_limits)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(daily["qt_material"], vert=False)
axes[0].set_title("Boxplot — demanda diária")
axes[0].set_xlabel("qt_material")

axes[1].boxplot(daily["receita_aprovada"], vert=False)
axes[1].set_title("Boxplot — receita diária")
axes[1].set_xlabel("receita_aprovada")

fig.tight_layout()
plt.show()

### Como interpretar estes boxplots

- **Caixa:** vai de Q1 (25% dos dias abaixo) até Q3 (75% dos dias abaixo). Portanto contém os 50% centrais da distribuição.
- **Linha laranja dentro da caixa:** mediana, isto é, metade dos dias abaixo e metade acima.
- **Bigodes:** chegam até a observação mais extrema que ainda esteja dentro dos limites de Tukey.
- **Círculos fora dos bigodes:** dias sinalizados como outliers estatísticos. Cada círculo é um dia.

Pelos limites calculados acima para a demanda: Q1 = **22.519 itens**, Q3 = **40.919 itens** e limite superior ≈ **68.519 itens**. Como o limite inferior é negativo (≈ -5.081), ele não é relevante para uma quantidade vendida não negativa. Assim, os **26 pontos** de demanda sinalizados são essencialmente dias acima de ~68,5 mil itens.

Para receita: Q1 ≈ **R$ 1,35 mi**, Q3 ≈ **R$ 2,25 mi** e limite superior ≈ **R$ 3,59 mi**. Existem **25 dias** sinalizados. A grande concentração de pontos à direita indica uma distribuição assimétrica, típica de e-commerce com campanhas e datas comerciais.

**Importante:** o ponto muito distante à direita não deve ser removido automaticamente. Ele representa um dia real que precisa ser confrontado com calendário comercial/campanhas.

In [ ]:
sql_top_days = f"""
WITH calendario_comercial AS (
  SELECT *
  FROM UNNEST([
    STRUCT(DATE '2025-11-28' AS data_evento, 'Black Friday' AS evento),
    STRUCT(DATE '2025-12-01', 'Cyber Monday'),
    STRUCT(DATE '2025-12-24', 'Véspera de Natal'),
    STRUCT(DATE '2025-12-25', 'Natal'),
    STRUCT(DATE '2026-01-01', 'Ano Novo'),
    STRUCT(DATE '2026-02-16', 'Carnaval'),
    STRUCT(DATE '2026-02-17', 'Carnaval'),
    STRUCT(DATE '2026-03-15', 'Dia do Consumidor'),
    STRUCT(DATE '2026-04-03', 'Paixão de Cristo'),
    STRUCT(DATE '2026-04-05', 'Páscoa'),
    STRUCT(DATE '2026-05-01', 'Dia do Trabalho'),
    STRUCT(DATE '2026-05-10', 'Dia das Mães'),
    STRUCT(DATE '2026-06-04', 'Corpus Christi'),
    STRUCT(DATE '2026-06-12', 'Dia dos Namorados')
  ])
),
daily AS (
  SELECT
    DATE(dt_hr_venda) AS data,
    SUM(qt_material) AS itens,
    SUM(nr_pedidos) AS pedidos,
    CAST(SUM(receita_aprovada) AS FLOAT64) AS receita,
    SAFE_DIVIDE(
      CAST(SUM(receita_aprovada) AS FLOAT64),
      SUM(qt_material)
    ) AS preco_medio_item
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1
),
top_days AS (
  SELECT *
  FROM daily
  ORDER BY itens DESC
  LIMIT 10
)
SELECT
  t.*,
  ARRAY_TO_STRING(
    ARRAY_AGG(
      IF(
        ABS(DATE_DIFF(t.data, c.data_evento, DAY)) <= 3,
        CONCAT(
          c.evento,
          ' (',
          CAST(DATE_DIFF(t.data, c.data_evento, DAY) AS STRING),
          'd)'
        ),
        NULL
      )
      IGNORE NULLS
    ),
    ', '
  ) AS evento_proximo
FROM top_days t
LEFT JOIN calendario_comercial c
  ON ABS(DATE_DIFF(t.data, c.data_evento, DAY)) <= 3
GROUP BY
  t.data,
  t.itens,
  t.pedidos,
  t.receita,
  t.preco_medio_item
ORDER BY t.itens DESC
"""

top_days = run_bq(sql_top_days, "top 10 dias + calendário comercial")
top_days["data"] = pd.to_datetime(top_days["data"])
display(top_days)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily["data"], daily["qt_material"], alpha=0.55, label="Demanda diária")
ax.scatter(
    top_days["data"],
    top_days["itens"],
    s=55,
    label="Top 10 dias",
    zorder=3,
)

for _, row in top_days.iterrows():
    label = row["data"].strftime("%d/%m")
    if pd.notna(row["evento_proximo"]) and row["evento_proximo"]:
        label += f"\n{row['evento_proximo']}"

    ax.annotate(
        label,
        (row["data"], row["itens"]),
        xytext=(5, 8),
        textcoords="offset points",
        fontsize=8,
        rotation=25,
    )

ax.set_title("Top dias de demanda na linha do tempo")
ax.set_xlabel("Data")
ax.set_ylabel("Itens vendidos")
ax.legend()
fig.tight_layout()
plt.show()

O calendário comercial acima é uma **camada contextual de apoio**. Ele permite verificar proximidade de até ±3 dias entre picos e datas relevantes. Em produção, essa informação deveria vir de uma dimensão/calendário versionada, e não ficar hardcoded no notebook.

# 4. Insights pedidos no enunciado

## 4.1 Preço médio de venda

Usamos média **ponderada**: `SUM(receita_aprovada) / SUM(qt_material)`. Uma média simples dos preços médios por linha daria o mesmo peso a volumes muito diferentes.

In [ ]:
sql_kpis = f"""
SELECT
  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita_total,
  SUM(nr_pedidos) AS pedidos,
  SUM(qt_material) AS itens,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(qt_material)) AS preco_medio_item,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(nr_pedidos)) AS ticket_medio,
  SAFE_DIVIDE(SUM(qt_material), SUM(nr_pedidos)) AS itens_por_pedido
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
"""

kpis = run_bq(sql_kpis, "KPIs gerais")
display(kpis)

### Por canal

In [ ]:
sql_channel = f"""
SELECT
  des_canal_venda_final_agrup AS canal,
  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita,
  SUM(nr_pedidos) AS pedidos,
  SUM(qt_material) AS itens,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(qt_material)) AS preco_medio_item,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(nr_pedidos)) AS ticket_medio,
  SAFE_DIVIDE(
    CAST(SUM(receita_aprovada) AS FLOAT64),
    SUM(CAST(SUM(receita_aprovada) AS FLOAT64)) OVER ()
  ) AS share_receita
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
GROUP BY 1
ORDER BY receita DESC
"""

channel = run_bq(sql_channel, "KPIs por canal")
display(channel)

### Por categoria válida

In [ ]:
sql_category = f"""
SELECT
  des_categoria_material AS categoria,
  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita,
  SUM(nr_pedidos) AS pedidos,
  SUM(qt_material) AS itens,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(qt_material)) AS preco_medio_item,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(nr_pedidos)) AS ticket_medio,
  SAFE_DIVIDE(
    CAST(SUM(receita_aprovada) AS FLOAT64),
    SUM(CAST(SUM(receita_aprovada) AS FLOAT64)) OVER ()
  ) AS share_receita
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
  AND SAFE_CAST(des_categoria_material AS NUMERIC) IS NULL
GROUP BY 1
ORDER BY receita DESC
"""

category = run_bq(sql_category, "KPIs por categoria")
display(category)

fig, ax = plt.subplots(figsize=(10, 5))
ordered = category.sort_values("preco_medio_item")
ax.barh(ordered["categoria"], ordered["preco_medio_item"])
ax.set_title("Preço médio por item — categoria")
ax.set_xlabel("Preço médio ponderado")
fig.tight_layout()
plt.show()

### Evolução mensal

In [ ]:
sql_monthly = f"""
SELECT
  FORMAT_DATE('%Y-%m', DATE(dt_hr_venda)) AS mes,
  CAST(SUM(receita_aprovada) AS FLOAT64) AS receita,
  SUM(nr_pedidos) AS pedidos,
  SUM(qt_material) AS itens,
  SAFE_DIVIDE(CAST(SUM(receita_aprovada) AS FLOAT64), SUM(qt_material)) AS preco_medio_item,
  COUNT(DISTINCT DATE(dt_hr_venda)) AS dias,
  SAFE_DIVIDE(SUM(qt_material), COUNT(DISTINCT DATE(dt_hr_venda))) AS demanda_media_diaria
FROM `{TABLE_ID}`
WHERE {DATE_FILTER}
GROUP BY 1
ORDER BY 1
"""

monthly = run_bq(sql_monthly, "KPIs mensais")
display(monthly)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly["mes"], monthly["preco_medio_item"], marker="o")
ax.set_title("Evolução mensal do preço médio por item")
ax.set_xlabel("Mês")
ax.set_ylabel("Preço médio")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 4.2 Potencial de vendas por dia da semana

Comparamos **média e mediana**. A mediana reduz o efeito de datas promocionais extremas.

In [ ]:
sql_weekday = f"""
WITH daily AS (
  SELECT
    DATE(dt_hr_venda) AS data,

    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 0
      WHEN 3 THEN 1
      WHEN 4 THEN 2
      WHEN 5 THEN 3
      WHEN 6 THEN 4
      WHEN 7 THEN 5
      WHEN 1 THEN 6
    END AS dia_semana_num,

    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 'Segunda'
      WHEN 3 THEN 'Terça'
      WHEN 4 THEN 'Quarta'
      WHEN 5 THEN 'Quinta'
      WHEN 6 THEN 'Sexta'
      WHEN 7 THEN 'Sábado'
      WHEN 1 THEN 'Domingo'
    END AS dia_semana,

    SUM(qt_material) AS itens,
    SUM(nr_pedidos) AS pedidos,
    CAST(SUM(receita_aprovada) AS FLOAT64) AS receita

  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1, 2, 3
),
global AS (
  SELECT
    APPROX_QUANTILES(itens, 100)[OFFSET(50)] AS mediana_global_itens
  FROM daily
)
SELECT
  d.dia_semana_num,
  d.dia_semana,
  COUNT(*) AS dias,
  AVG(d.itens) AS itens_media,
  APPROX_QUANTILES(d.itens, 100)[OFFSET(50)] AS itens_mediana,
  AVG(d.pedidos) AS pedidos_media,
  AVG(d.receita) AS receita_media,
  SAFE_DIVIDE(
    APPROX_QUANTILES(d.itens, 100)[OFFSET(50)],
    g.mediana_global_itens
  ) AS indice_potencial_itens
FROM daily d
CROSS JOIN global g
GROUP BY 1, 2, g.mediana_global_itens
ORDER BY 1
"""

weekday = run_bq(sql_weekday, "potencial por dia da semana")
display(weekday)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(weekday["dia_semana"], weekday["itens_mediana"])
ax.set_title("Potencial de demanda por dia da semana")
ax.set_xlabel("Dia da semana")
ax.set_ylabel("Mediana de itens/dia")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

### Qual dia vence em cada semana?

O gráfico anterior compara a distribuição agregada por dia da semana. Agora olhamos **semana a semana** e identificamos qual dia teve maior demanda.

A linha tracejada mostra a **mediana da posição do dia vencedor**. Porém, para responder qual dia vence mais frequentemente, a estatística correta é a **moda/frequência**, mostrada logo depois.

In [ ]:
sql_weekly_winners = f"""
WITH daily AS (
  SELECT
    DATE(dt_hr_venda) AS data,
    DATE_TRUNC(DATE(dt_hr_venda), WEEK(MONDAY)) AS semana_inicio,

    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 0
      WHEN 3 THEN 1
      WHEN 4 THEN 2
      WHEN 5 THEN 3
      WHEN 6 THEN 4
      WHEN 7 THEN 5
      WHEN 1 THEN 6
    END AS dia_semana_num,

    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 'Segunda'
      WHEN 3 THEN 'Terça'
      WHEN 4 THEN 'Quarta'
      WHEN 5 THEN 'Quinta'
      WHEN 6 THEN 'Sexta'
      WHEN 7 THEN 'Sábado'
      WHEN 1 THEN 'Domingo'
    END AS dia_semana,

    SUM(qt_material) AS itens

  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1, 2, 3, 4
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY semana_inicio
      ORDER BY itens DESC, data
    ) AS rn
  FROM daily
)
SELECT
  semana_inicio,
  data,
  dia_semana_num,
  dia_semana,
  itens
FROM ranked
WHERE rn = 1
ORDER BY semana_inicio
"""

weekly_winners = run_bq(
    sql_weekly_winners,
    "dia vencedor por semana",
)
weekly_winners["semana_inicio"] = pd.to_datetime(weekly_winners["semana_inicio"])
weekly_winners["data"] = pd.to_datetime(weekly_winners["data"])

weekday_labels = [
    "Segunda",
    "Terça",
    "Quarta",
    "Quinta",
    "Sexta",
    "Sábado",
    "Domingo",
]

median_winner = float(weekly_winners["dia_semana_num"].median())

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(
    weekly_winners["semana_inicio"],
    weekly_winners["dia_semana_num"],
    s=45,
)
ax.plot(
    weekly_winners["semana_inicio"],
    weekly_winners["dia_semana_num"],
    alpha=0.3,
)
ax.axhline(
    median_winner,
    linestyle="--",
    label=f"Mediana da posição = {median_winner:.1f}",
)
ax.set_yticks(range(7))
ax.set_yticklabels(weekday_labels)
ax.set_title("Dia de maior demanda em cada semana")
ax.set_xlabel("Semana iniciada em")
ax.set_ylabel("Dia vencedor")
ax.legend()
fig.tight_layout()
plt.show()

winner_frequency = (
    weekly_winners["dia_semana"]
    .value_counts()
    .reindex(weekday_labels, fill_value=0)
    .rename_axis("dia_semana")
    .reset_index(name="semanas_vencedoras")
)

winner_frequency["pct_semanas"] = (
    winner_frequency["semanas_vencedoras"]
    / winner_frequency["semanas_vencedoras"].sum()
)

display(winner_frequency)

winner_mode = winner_frequency.loc[
    winner_frequency["semanas_vencedoras"].idxmax(),
    "dia_semana",
]
print(f"Dia que mais venceu semanas (moda): {winner_mode}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    winner_frequency["dia_semana"],
    winner_frequency["semanas_vencedoras"],
)
ax.set_title("Frequência do dia de maior demanda por semana")
ax.set_xlabel("Dia da semana")
ax.set_ylabel("Quantidade de semanas em que foi o maior dia")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()

## 4.3 Flutuação ao longo do dia

Para evitar que dias de grande volume dominem a análise, calculamos a **participação de cada hora no total de seu próprio dia** e só depois fazemos a média entre os dias.

In [ ]:
sql_hourly = f"""
WITH hourly_daily AS (
  SELECT
    DATE(dt_hr_venda) AS data,
    EXTRACT(HOUR FROM dt_hr_venda) AS hora,
    CAST(SUM(receita_aprovada) AS FLOAT64) AS receita,
    SUM(nr_pedidos) AS pedidos,
    SUM(qt_material) AS itens
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1, 2
),
shares AS (
  SELECT
    *,
    SAFE_DIVIDE(receita, SUM(receita) OVER (PARTITION BY data)) AS share_receita,
    SAFE_DIVIDE(itens, SUM(itens) OVER (PARTITION BY data)) AS share_itens
  FROM hourly_daily
)
SELECT
  hora,
  AVG(receita) AS receita_media,
  AVG(pedidos) AS pedidos_media,
  AVG(itens) AS itens_media,
  AVG(share_receita) AS share_receita_media,
  AVG(share_itens) AS share_itens_media
FROM shares
GROUP BY 1
ORDER BY 1
"""

hourly = run_bq(sql_hourly, "perfil intradiário")
display(hourly)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly["hora"], hourly["share_itens_media"] * 100, marker="o")
ax.set_title("Participação média da hora na demanda diária")
ax.set_xlabel("Hora")
ax.set_ylabel("% médio dos itens do dia")
ax.set_xticks(range(24))
fig.tight_layout()
plt.show()

### Heatmap dia da semana × hora

In [ ]:
sql_hour_dow = f"""
WITH base AS (
  SELECT
    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 0
      WHEN 3 THEN 1
      WHEN 4 THEN 2
      WHEN 5 THEN 3
      WHEN 6 THEN 4
      WHEN 7 THEN 5
      WHEN 1 THEN 6
    END AS dia_semana_num,

    CASE EXTRACT(DAYOFWEEK FROM DATE(dt_hr_venda))
      WHEN 2 THEN 'Segunda'
      WHEN 3 THEN 'Terça'
      WHEN 4 THEN 'Quarta'
      WHEN 5 THEN 'Quinta'
      WHEN 6 THEN 'Sexta'
      WHEN 7 THEN 'Sábado'
      WHEN 1 THEN 'Domingo'
    END AS dia_semana,

    EXTRACT(HOUR FROM dt_hr_venda) AS hora,
    SUM(qt_material) AS itens
  FROM `{TABLE_ID}`
  WHERE {DATE_FILTER}
  GROUP BY 1, 2, 3
),
share AS (
  SELECT
    *,
    SAFE_DIVIDE(
      itens,
      SUM(itens) OVER (PARTITION BY dia_semana_num)
    ) AS share
  FROM base
)
SELECT *
FROM share
ORDER BY dia_semana_num, hora
"""

hour_dow = run_bq(sql_hour_dow, "dia da semana x hora")

heat = (
    hour_dow.pivot(index="dia_semana", columns="hora", values="share")
    .reindex(["Segunda", "Terça", "Quarta", "Quinta", "Sexta", "Sábado", "Domingo"])
)

fig, ax = plt.subplots(figsize=(14, 5))
image = ax.imshow(heat.to_numpy() * 100, aspect="auto")
ax.set_title("Distribuição da demanda por hora e dia da semana")
ax.set_xlabel("Hora")
ax.set_ylabel("Dia da semana")
ax.set_xticks(range(24))
ax.set_xticklabels(range(24))
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
cbar = fig.colorbar(image, ax=ax)
cbar.set_label("% da demanda do respectivo dia da semana")
fig.tight_layout()
plt.show()

# 5. Relações entre métricas — Python

A partir daqui usamos a pequena visão diária já agregada pelo BigQuery. Correlação é uma ferramenta exploratória, não causalidade.

In [ ]:
corr_columns = [
    "receita_aprovada",
    "nr_pedidos",
    "qt_material",
    "vlr_venda_desconto",
    "preco_medio_item",
    "ticket_medio",
    "itens_por_pedido",
]

corr = daily[corr_columns].corr(method="spearman")
display(corr)

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(corr.to_numpy(), vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)
ax.set_title("Correlação Spearman — métricas diárias")

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

# 6. Model Readiness

### Target proposto
**`qt_material` diário**, por representar diretamente a demanda operacional em unidades.

`receita_aprovada` permanece como KPI financeiro e pode ser modelada em uma análise complementar.

### Risco de leakage
Para prever o dia *t*, métricas realizadas do próprio dia *t* não podem ser features: receita, pedidos, itens, desconto e mix realizado do mesmo período só existem depois da venda.

### Features candidatas
- calendário: dia da semana, mês, fim de semana;
- lags `t-1`, `t-7`, `t-14`, `t-28`;
- rolling mean/std calculados apenas com passado;
- lags de preço médio, receita e pedidos;
- tendência temporal;
- feriados/datas comerciais, caso sejam adicionados como fonte conhecida antes da previsão.

### Validação
Split aleatório não é apropriado. Usaremos **backtest temporal / expanding window**.

### Baselines obrigatórios
- naive t-1;
- naive sazonal t-7;
- média móvel histórica.

### Métricas
MAE, WAPE, Bias e Skill vs baseline. MAPE será tratado com cautela.

# 7. Resumo executivo

In [ ]:
top_day = top_days.iloc[0]
top_weekday = weekday.loc[weekday["itens_mediana"].idxmax()]
bottom_weekday = weekday.loc[weekday["itens_mediana"].idxmin()]
peak_hour = hourly.loc[hourly["share_itens_media"].idxmax()]

executive_summary = pd.DataFrame([
    {
        "tema": "Data Quality",
        "insight": (
            f"{float(invalid_category.loc[0, 'pct_categoria_invalida']):.2%} "
            "das linhas possuem categoria inválida."
        ),
        "implicacao": "Preservar vendas nos totais e excluir apenas de análises categóricas.",
    },
    {
        "tema": "Receita negativa",
        "insight": (
            f"{float(negative_revenue.loc[0, 'pct_linhas_negativas']):.2%} "
            "das linhas têm receita negativa."
        ),
        "implicacao": "Validar estorno/devolução/ajuste antes de qualquer tratamento.",
    },
    {
        "tema": "Maior dia de demanda",
        "insight": f"{top_day['data']} — {int(top_day['itens']):,} itens.",
        "implicacao": "Picos legítimos devem ser aprendidos pelo modelo, não removidos mecanicamente.",
    },
    {
        "tema": "Dia da semana",
        "insight": (
            f"Maior mediana: {top_weekday['dia_semana']}; "
            f"menor: {bottom_weekday['dia_semana']}."
        ),
        "implicacao": "Dia da semana é uma feature relevante.",
    },
    {
        "tema": "Intradiário",
        "insight": f"Hora de maior participação média: {int(peak_hour['hora'])}h.",
        "implicacao": "Existe padrão horário relevante para planejamento operacional.",
    },
])

display(executive_summary)